# Advanced Hybrid AE-RF Network Intrusion Detection Model Training

This notebook acts as the Google Colab environment designed to train the Hybrid Deep Autoencoder + Random Forest (AE-RF) model for the Sentinel IDS.
It leverages Google's GPU/TPU resources for the computationally intensive autoencoder training phase using Keras/TensorFlow, before serializing the pipeline using `joblib`.

**Output Artifact:** `model.joblib` (Ready for Antigravity Production Backend)

In [ ]:
!pip install scikit-learn pandas numpy joblib tensorflow keras

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

## Dataset Preparation (NSL-KDD / UNSW-NB15 Simulation)
In a real Colab runtime, you would mount Google Drive and load a multi-gigabyte dataset. For this artifact generation, we synthesize high-dimensional traffic data.

In [ ]:
def load_dataset(n_samples=50000):
    print(f"Synthesizing {n_samples} samples of high-dimensional network telemetry...")
    y = np.random.choice([0, 1], size=n_samples, p=[0.9, 0.1]) # 10% anomalies
    X = np.zeros((n_samples, 3))
    for i in range(n_samples):
        if y[i] == 0: # Normal Traffic
            X[i, 0] = np.random.normal(500, 100)   # bytes
            X[i, 1] = np.random.normal(0.5, 0.2)   # duration
            X[i, 2] = np.random.poisson(0.1)       # failed logins
        else: # Malicious Traffic (Data Exfil, Brute Force, etc.)
            X[i, 0] = np.random.normal(5000, 2000) 
            X[i, 1] = np.random.normal(5.0, 2.0)   
            X[i, 2] = np.random.poisson(5.0)       
    
    # Add significant variance to simulate real network noise
    X += np.random.normal(0, 0.5, X.shape)
    
    df = pd.DataFrame(X, columns=['bytes_transferred', 'connection_duration', 'failed_logins'])
    return df, y

X, y = load_dataset()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training Set: {X_train.shape}")

## Deep Autoencoder (Keras Implementation)
Training the non-linear Autoencoder bottleneck to extract latent space representations and reconstruction error.

In [ ]:
# Note: We wrap the Keras model in a scikit-learn BaseEstimator for seamless Joblib Pipeline serialization.

class KerasAutoEncoderFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, input_dim=3, latent_dim=2, epochs=20, batch_size=256):
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.epochs = epochs
        self.batch_size = batch_size
        self.autoencoder = None
        
    def _build_model(self):
        input_layer = Input(shape=(self.input_dim,))
        # Encoder
        encoded = Dense(16, activation='relu')(input_layer)
        encoded = Dense(8, activation='relu')(encoded)
        latent = Dense(self.latent_dim, activation='relu', name='bottleneck')(encoded)
        
        # Decoder
        decoded = Dense(8, activation='relu')(latent)
        decoded = Dense(16, activation='relu')(decoded)
        output_layer = Dense(self.input_dim, activation='linear')(decoded)
        
        autoencoder = Model(inputs=input_layer, outputs=output_layer)
        autoencoder.compile(optimizer='adam', loss='mse')
        return autoencoder

    def fit(self, X, y=None):
        # Standardize for Neural Network
        X_num = X.values if isinstance(X, pd.DataFrame) else X
        
        # We only train the Autoencoder on NORMAL traffic (Unsupervised)
        # Assuming 'y' is available during fit to filter true normal traffic
        if y is not None:
            X_train_normal = X_num[y == 0]
        else:
            X_train_normal = X_num
            
        self.autoencoder = self._build_model()
        
        print("Training Deep Autoencoder on Normal Baselines...")
        # In Colab this utilizes the GPU
        self.autoencoder.fit(X_train_normal, X_train_normal, 
                             epochs=self.epochs, 
                             batch_size=self.batch_size,
                             shuffle=True, 
                             verbose=0)
        return self
        
    def transform(self, X):
        X_num = X.values if isinstance(X, pd.DataFrame) else X
        
        # 1. Get Reconstruction Error (MSE)
        reconstructed = self.autoencoder.predict(X_num, verbose=0)
        mse = np.mean(np.square(X_num - reconstructed), axis=1).reshape(-1, 1)
        
        # Combine original features with the derived Latent MSE
        return np.hstack([X_num, mse])
        
    # For Joblib picking of Keras models, custom state management is required
    def __getstate__(self):
        state = self.__dict__.copy()
        if self.autoencoder is not None:
            # Serialize the weights temporarily
            state['keras_weights'] = self.autoencoder.get_weights()
            state['autoencoder'] = None
        return state
        
    def __setstate__(self, state):
        self.__dict__.update(state)
        if 'keras_weights' in state:
            self.autoencoder = self._build_model()
            self.autoencoder.set_weights(state['keras_weights'])
            del self.__dict__['keras_weights']

## Random Forest Classification Pipeline
Chaining the Deep Autoencoder with `RandomForestClassifier` for deterministic attack categorization.

In [ ]:
print("Constructing Full ML Pipeline...")
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('autoencoder_latent', KerasAutoEncoderFeatureExtractor(epochs=10)),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1))
])

pipeline.fit(X_train, y_train)

print("Evaluating Hybrid Model...")
train_score = pipeline.score(X_train, y_train)
test_score = pipeline.score(X_test, y_test)

print(f"Training Accuracy: {train_score * 100:.2f}%")
print(f"Validation Accuracy: {test_score * 100:.2f}%")

## Exporting to Antigravity
Serializing the whole fitted pipeline via `joblib` so Antigravity backend can serve real-time FastApi inference seamlessly.

In [ ]:
artifact = {
    'model': pipeline,
    'features': X.columns.tolist(),
    'version': '2.0.0-gpu-colab',
    'description': 'Keras AE + Sklearn RF Hybrid Model'
}

import joblib
joblib.dump(artifact, 'model.joblib')
print("Successfully exported model.joblib! Download this artifact and place it in the /backend folder of the Antigravity IDE.")